In [1]:
import os

In [2]:
%pwd

'd:\\Project\\creditcard-fraud-detection-proj\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Project\\creditcard-fraud-detection-proj'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    eval_metric: str
    target_column: str

In [6]:
from cred_card_proj.constants import *
from cred_card_proj.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.XGBoost
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            eval_metric = params.eval_metric,
            target_column = schema.name
            
        )

        return model_trainer_config

In [8]:
import pandas as pd
import os
from cred_card_proj import logger
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib

In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    
    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]

        sm = SMOTE(random_state=42)
        train_x_res, train_y_res = sm.fit_resample(train_x, train_y)

        xgb_model  = XGBClassifier(eval_metric = self.config.eval_metric, random_state = 42)
        xgb_model.fit(train_x_res, train_y_res)

        joblib.dump(xgb_model, os.path.join(self.config.root_dir, self.config.model_name))

In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-08-11 16:29:45,584: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-11 16:29:45,586: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-11 16:29:45,589: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-08-11 16:29:45,591: INFO: common: created directory at: artifacts]
[2026-08-11 16:29:45,593: INFO: common: created directory at: artifacts/model_trainer]
